# 03. Model Training & Hyperparameter Tuning
## House Price Prediction System

In this notebook, we train and optimize 8+ Machine Learning and Deep Learning models:
1. **Linear Models**: Linear Regression (Baseline), Ridge ($L_2$), Lasso ($L_1$)
2. **Tree Models**: Decision Tree Regressor, Random Forest Regressor ($N=1200, Depth=60$)
3. **Boosting Models**: Gradient Boosting Regressor (GBR), XGBoost Regressor
4. **Neural Models**: Multi-Layer Perceptron (MLP), LSTM Tabular Regressor
5. **5-Fold Cross-Validation** and Hyperparameter Tuning with `GridSearchCV`.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_raw_data
from src.preprocessing import remove_outliers, log_transform_target, create_full_data_pipeline
from src.models import get_all_models, get_model
from src.evaluate import evaluate_single_model, save_results_to_csv
from src.config import RANDOM_SEED, TEST_SIZE, TARGET_COLUMN

### 1. Load Data and Process Features

In [ ]:
train_raw, _ = load_raw_data()
train_clean = remove_outliers(train_raw)
train_split, test_split = train_test_split(train_clean, test_size=TEST_SIZE, random_state=RANDOM_SEED)

pipeline, X_train_proc_df, y_train_log = create_full_data_pipeline(train_split, scale_numeric=True)
X_test_raw = test_split.drop(columns=[TARGET_COLUMN, 'Id'], errors='ignore')
y_test_dollars = test_split[TARGET_COLUMN].values
y_test_log = log_transform_target(y_test_dollars)
X_test_proc = pipeline.transform(X_test_raw)
X_train_proc = X_train_proc_df.values

print(f"Training instances: {X_train_proc.shape[0]}, Features: {X_train_proc.shape[1]}")
print(f"Test instances:     {X_test_proc.shape[0]}")

### 2. Fit and Evaluate All Models with 5-Fold Cross Validation

In [ ]:
models = get_all_models(fast_mode=False)
results = []

for name, model in models.items():
    print(f"Training & Cross-Validating {name}...")
    metrics = evaluate_single_model(
        model_name=name,
        model=model,
        X_train=X_train_proc,
        y_train_log=y_train_log,
        X_test=X_test_proc,
        y_test_log=y_test_log,
        run_cv=True,
    )
    results.append(metrics)

results_df = pd.DataFrame(results)
display(results_df)